In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Qedma의 Qiskit Function
*[API 참조](https://docs.quantum.ibm.com/api/functions/qedma-qesem)를 참조하세요*

> **Note:** Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게만 제공되는 실험적 기능입니다. 미리 보기 출시 상태에 있으며 변경될 수 있습니다.

## 개요
양자 처리 장치는 최근 몇 년간 크게 개선되었지만, 기존 하드웨어의 노이즈와 불완전성으로 인한 오류는 양자 알고리즘 개발자에게 여전히 중심적인 과제입니다. 고전적으로 검증할 수 없는 유틸리티 규모 양자 계산에 가까워짐에 따라, 보장된 정확도로 노이즈를 제거하는 솔루션이 점점 더 중요해지고 있습니다. 이 과제를 극복하기 위해 Qedma는 IBM Quantum Platform에 [Qiskit Function](/guides/functions)으로 원활하게 통합된 양자 오류 완화(QESEM)를 개발했습니다.

QESEM을 사용하면 사용자가 노이즈가 있는 QPU에서 양자 Circuit을 실행하여 근본적인 한계에 가까운 매우 효율적인 QPU 시간 오버헤드로 매우 정확한 오류 없는 결과를 얻을 수 있습니다. 이를 달성하기 위해 QESEM은 오류의 특성화 및 감소를 위해 Qedma가 개발한 독점 방법 모음을 활용합니다. 오류 감소 기법에는 게이트 최적화, 노이즈 인식 트랜스파일, 오류 억제(ES), 편향 없는 오류 완화(EM)가 포함됩니다. 이 특성화 기반 방법들의 조합으로 사용자는 일반적인 대용량 양자 Circuit에서 신뢰할 수 있고 오류 없는 결과를 얻을 수 있으며, 그렇지 않으면 달성할 수 없는 응용 프로그램을 실현할 수 있습니다.

기본 구성 요소에 대한 전체 설명과 유틸리티 규모 시연은 논문 [유틸리티 규모 양자 Circuit을 위한 신뢰할 수 있는 고정확도 오류 완화](https://arxiv.org/abs/2508.10997)를 참조하세요.
## 설명
Qedma의 QESEM 함수를 사용하여 오류 억제 및 완화로 Circuit을 쉽게 추정하고 실행하여 더 큰 Circuit 볼륨과 더 높은 정확도를 달성할 수 있습니다. QESEM을 사용하려면 양자 Circuit, 측정할 Observable 집합, 각 Observable에 대한 목표 통계 정확도, 선택한 QPU를 제공합니다. 목표 정확도로 Circuit을 실행하기 전에 Circuit 실행이 필요 없는 분석적 계산을 기반으로 필요한 QPU 시간을 추정할 수 있습니다. QPU 시간 추정에 만족하면 QESEM으로 Circuit을 실행할 수 있습니다.

Circuit을 실행하면 QESEM은 Circuit에 맞춤화된 장치 특성화 프로토콜을 실행하여 Circuit에서 발생하는 오류에 대한 신뢰할 수 있는 노이즈 모델을 생성합니다. 특성화를 기반으로 QESEM은 먼저 노이즈 인식 트랜스파일을 구현하여 입력 Circuit을 목표 Observable에 영향을 미치는 노이즈를 최소화하는 물리적 Qubit과 게이트 집합으로 매핑합니다. 여기에는 기본적으로 사용 가능한 게이트(IBM&reg; 장치의 CX/CZ)와 QESEM이 최적화한 추가 게이트를 포함하는 QESEM의 확장된 게이트 집합이 포함됩니다. 그런 다음 QESEM은 QPU에서 특성화 기반 ES 및 EM Circuit 집합을 실행하고 측정 결과를 수집합니다. 이 결과들은 고전적으로 후처리되어 요청된 정확도에 해당하는 각 Observable에 대한 편향 없는 기댓값과 오차 막대를 제공합니다.

![Qedma QESEM 개요](../docs/images/guides/qedma-qesem/overview.svg)
QESEM은 다양한 양자 응용 프로그램과 오늘날 달성 가능한 가장 큰 Circuit 볼륨에서 높은 정확도의 결과를 제공하는 것으로 입증되었습니다. QESEM은 아래 벤치마크 섹션에서 시연된 다음과 같은 사용자 지향 기능을 제공합니다:
-	**보장된 정확도:** QESEM은 Observable의 기댓값에 대한 편향 없는 추정값을 출력합니다. EM 방법은 이론적 보장을 갖추고 있으며, Qedma의 최첨단 특성화와 함께 완화가 사용자 지정 정확도까지 노이즈 없는 Circuit 출력으로 수렴하도록 보장합니다. 체계적인 오류나 편향이 발생하기 쉬운 많은 휴리스틱 EM 방법과 달리, QESEM의 보장된 정확도는 일반적인 양자 Circuit과 Observable에서 신뢰할 수 있는 결과를 보장하는 데 필수적입니다.
-	**대규모 QPU로의 확장성:** QESEM의 QPU 시간은 Circuit 볼륨에 따라 달라지지만, 그 외에는 qubit 수와 무관합니다. Qedma는 IBM Quantum 127 qubit Eagle 및 133 qubit Heron 장치를 포함하여 오늘날 사용 가능한 가장 큰 양자 장치에서 QESEM을 시연했습니다.
-	**응용 프로그램 무관:** QESEM은 해밀토니안 시뮬레이션, VQE, QAOA, 진폭 추정을 포함한 다양한 응용 프로그램에서 시연되었습니다. 사용자는 임의의 양자 Circuit과 측정할 Observable을 입력하고 정확한 오류 없는 결과를 얻을 수 있습니다. 유일한 제한은 접근 가능한 Circuit 볼륨과 출력 정확도를 결정하는 하드웨어 사양과 할당된 QPU 시간에 의해 결정됩니다. 대조적으로, 많은 오류 감소 솔루션은 응용 프로그램에 특화되거나 제어되지 않는 휴리스틱을 포함하여 일반적인 양자 Circuit과 응용 프로그램에 적용할 수 없습니다.
-  **확장된 게이트 집합:** QESEM은 분수 각도 게이트를 지원하고 IBM Quantum Heron 및 Eagle 장치에서 Qedma 최적화 분수 각도 $Rzz(\theta)$ 게이트를 제공합니다. 이 확장된 게이트 집합은 더 효율적인 컴파일을 가능하게 하고 기본 CX/CZ 컴파일에 비해 최대 2배 더 큰 Circuit 볼륨을 실현합니다.
-	**Multibase Observable:** QESEM은 일반 해밀토니안과 같이 많은 비교환 Pauli 문자열로 구성된 입력 Observable을 지원합니다. 측정 기저의 선택과 QPU 리소스 할당(샷 및 Circuit)의 최적화는 요청된 정확도를 위한 필요한 QPU 시간을 최소화하기 위해 QESEM에 의해 자동으로 수행됩니다. 하드웨어 충실도와 실행 속도를 고려하는 이 최적화를 통해 더 깊은 Circuit을 실행하고 더 높은 정확도를 얻을 수 있습니다.
## 벤치마크
QESEM은 다양한 사용 사례와 응용 프로그램에서 테스트되었습니다. 다음 예제는 QESEM으로 실행할 수 있는 워크로드 유형을 평가하는 데 도움이 됩니다.

주어진 Circuit과 Observable에 대한 오류 완화와 고전적 시뮬레이션의 어려움을 정량화하는 핵심 성과 지표는 **활성 볼륨**입니다: Circuit에서 Observable에 영향을 미치는 CNOT 게이트의 수. 활성 볼륨은 Circuit 깊이와 너비, Observable 가중치, Observable의 라이트콘을 결정하는 Circuit 구조에 따라 달라집니다. 자세한 내용은 [2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s) 강연을 참조하세요. QESEM은 고볼륨 레짐에서 특히 큰 가치를 제공하며 일반적인 Circuit과 Observable에 대한 신뢰할 수 있는 결과를 제공합니다.

![활성 볼륨](../docs/images/guides/qedma-qesem/active_volume.svg)

| 응용 프로그램    | qubit 수 | 장치 | Circuit 설명 | 정확도 | 총 시간 | 런타임 사용 |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE Circuit  | 8              | Eagle (r3) | 21개 총 레이어, 9개 측정 기저, 1D 체인                    | 98%      | 35분      | 14분         |
| Kicked Ising   | 28              | Eagle (r3) | 고유 레이어 3개 x 3단계, 2D heavy-hex 토폴로지                      | 97%     | 22분      | 4분          |
| Kicked Ising   | 28              | Eagle (r3) | 고유 레이어 3개 x 8단계, 2D heavy-hex 토폴로지                     | 97%      | 116분      | 23분          |
| 트로터화 해밀토니안 시뮬레이션   | 40  | Eagle (r3)            | 고유 레이어 2개 x 10 트로터 단계, 1D 체인                    | 97%      | 3시간     | 25분         |
| 트로터화 해밀토니안 시뮬레이션   | 119  | Eagle (r3)           | 고유 레이어 3개 x 9 트로터 단계, 2D heavy-hex 토폴로지                    | 95%      | 6.5시간     | 45분         |
| Kicked Ising  | 136             | Heron (r2) | 고유 레이어 3개 x 15단계, 2D heavy-hex 토폴로지                    | 99%      | 52분             | 9분           |

여기서 정확도는 Observable의 이상적인 값에 대한 상대적인 측정값입니다: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, 여기서 '$\epsilon$'은 완화의 절대 정밀도(사용자 입력으로 설정)이고, $\langle O \rangle_{ideal}$은 노이즈 없는 Circuit에서의 Observable 값입니다.
'런타임 사용'은 배치 모드에서의 벤치마크 사용량(개별 Job 사용량의 합계)을 측정하며, '총 시간'은 추가적인 고전 및 통신 시간을 포함하는 세션 모드에서의 사용량(실험 벽시계 시간)을 측정합니다. QESEM은 두 모드 모두에서 실행 가능하므로 사용자는 사용 가능한 리소스를 최대한 활용할 수 있습니다.

28 qubit Kicked Ising Circuit은 ibm_kawasaki의 세 개 연결 루프에서 Shinjo et al.이 연구한 이산 시간 준결정을 시뮬레이션합니다(참조 [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) 및 [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). 여기서 취한 Circuit 매개변수는 $(\theta_x, \theta_z) = (0.9 \pi, 0)$이며, 강자성 초기 상태 $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$입니다. 측정된 Observable은 자화의 절댓값 $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$입니다. 유틸리티 규모 Kicked Ising 실험은 ibm_fez의 136개 최고 Qubit에서 실행되었습니다. 이 특정 벤치마크는 Clifford 각도 $(\theta_x, \theta_z) = (\pi, 0)$에서 실행되었으며, 이 각도에서는 활성 볼륨이 Circuit 깊이에 따라 천천히 증가하여 높은 장치 충실도와 함께 짧은 런타임에 높은 정확도를 달성할 수 있습니다.

트로터화 해밀토니안 시뮬레이션 Circuit은 분수 각도의 횡방향 필드 이징 모델에 대한 것입니다: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ 및 $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$(참조 [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). 유틸리티 규모 Circuit은 ibm_brisbane의 119개 최고 Qubit에서 실행되었으며, 40 qubit 실험은 사용 가능한 최고의 체인에서 실행되었습니다. 정확도는 자화에 대해 보고됩니다. 더 높은 가중치의 Observable에 대해서도 높은 정확도의 결과를 얻었습니다.

VQE Circuit은 Deutsches Elektronen-Synchrotron(DESY)의 양자 기술 및 응용 센터 연구자들과 함께 개발되었습니다. 여기서 목표 Observable은 많은 수의 비교환 Pauli 문자열로 구성된 해밀토니안으로, 다중 기저 Observable에 대한 QESEM의 최적화된 성능을 강조합니다. 완화는 고전적으로 최적화된 앤자츠에 적용되었습니다. 이 결과들은 아직 미발표 상태이지만, 유사한 구조적 특성을 가진 다른 Circuit에서도 동일한 품질의 결과를 얻을 수 있습니다.
## 시작하기
[IBM Quantum Platform API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하고 다음과 같이 QESEM Qiskit Function을 선택하세요. (이 코드 스니펫은 이미 로컬 환경에 [계정을 저장](/guides/functions#install-qiskit-functions-catalog-client)했다고 가정합니다.)

In [ ]:
import qiskit
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

### Time estimation job example

The time estimation job is useful for estimating the QPU time required for a given `pub` and `backend_name`. `backend_name` can also be set to any simulator backend; for example, `fake_fez`.

QESEM uses a quasi-probabilistic, characterization-based EM method. This method has a QPU time overhead that roughly scales as:

$T_{QPU} = a \frac{e^{\alpha IF\cdot V_a}}{\epsilon^2} + b$

Where $V_a$ is the active volume of the circuit, $\epsilon$ is the target precision, and $IF$ is the infidelity of the native gates.

Note that `"estimate_time_only": "empirical"` uses a few minutes of QPU time to estimate the time required for the job (if the backend is a real device; if it's a simulator, then no QPU time is used). This will usually take around 5 minutes, but no more than 10 minutes. If the infidelity changes drastically between the empirical time estimation job and the mitigation job, the QPU time will also change drastically.

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
backend_name = "fake_fez"

circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "empirical",
    },
    backend_name=backend_name,  # example: "fake_fez", "ibm_fez"
)

In [4]:
time_estimate_result = (
    time_estimation_job.result()
)  # a list of results per pub (circuit)

다음 코드 스니펫은 QPU 시간 추정값을 검색하는 방법을 설명합니다(`estimate_time_only`가 설정된 경우):

In [106]:
pub_result = time_estimate_result[0]

print(
    f"The estimated QPU time for mitigation for this PUB is: {pub_result.metadata['time_estimation_sec']}"
)
print(
    f"The QPU time that this time estimation job took is (here it is 0 because we used fake_fez): {pub_result.metadata['total_qpu_time']}"
)
print(
    f"Gates fidelity measured during the experiment: {pub_result.metadata['gate_fidelities']}"
)
print(f"Total shots: {pub_result.metadata['total_shots']}")
print(f"Resource usage breakdown: {pub_result.metadata['resource_usage']}")

The estimated QPU time for mitigation for this PUB is: 300
The QPU time that this time estimation job took is (here it is 0 because we used fake_fez): 0
Gates fidelity measured during the experiment: {'CZ': 0.9951354916722668, 'ID1Q': 0.9991246627329172}
Total shots: 220000
Resource usage breakdown: {'RUNNING: MAPPING': {'CPU_TIME': 33.6066133165732, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 184.53575124032795, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}


다음 코드 스니펫은 완화 결과(`estimate_time_only`가 설정되지 않은 경우)와 실행 지표를 검색하는 방법을 보여줍니다. 이것들은 QESEM 실행에 다양한 매개변수가 미치는 영향에 대한 더 깊은 이해를 가능하게 하는 필수 데이터를 포함합니다. 연구를 기반으로 논문을 작성할 때도 관련이 있을 수 있습니다.

In [6]:
empirical_estimation_mitigation_results = time_estimate_result[0].metadata[
    "empirical_estimation_mitigation_results"
][0]  # a list per parameter

In [12]:
print("Partial results for the observables:")

print(
    f"    Mitigated expectation values: {empirical_estimation_mitigation_results.data.evs}"
)
print(
    f"    Mitigated error bars: {empirical_estimation_mitigation_results.data.stds}"
)
print(
    f"    Number of shots used for mitigation: {empirical_estimation_mitigation_results.metadata['mitigation_shots']}"
)
transpiled_circ = empirical_estimation_mitigation_results.metadata[
    "transpiled_circ"
]
print(f"    Qubit mapping: {transpiled_circ['qubit_maps']}")
print(
    f"    Number of measurement bases: {transpiled_circ['num_measurement_bases']}\n"
)

# results per obs
emp_obs_results = empirical_estimation_mitigation_results.metadata["results"][
    0
]
# print(f"Results for each observable: {results}")
print("Results for each observable:")

for i, (obs_array, result_dict) in enumerate(emp_obs_results):
    # obs_array, result_dict = results
    print(f"Observable {i+1}: {obs_array}")
    print(
        f"    QESEM mitigated value: {result_dict['qesem']['value']} \u00b1 {result_dict['qesem']['error_bar']}"
    )

Partial results for the observables:
    Mitigated expectation values: [1.00347302 1.00693905]
    Mitigated error bars: [0.00304061 0.00714276]
    Number of shots used for mitigation: 180000
    Qubit mapping: [[[0, 136], [1, 143], [2, 142], [3, 141], [4, 140]]]
    Number of measurement bases: 2

Results for each observable:
Observable 1: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
    QESEM mitigated value: 1.003473015776871 ± 0.0030406128032204015
Observable 2: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
    QESEM mitigated value: 1.0069390542613554 ± 0.0071427606736885925


### QESEM mitigation job example

The following example executes a QESEM job:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # example: "ibm_fez"
    # options = {
    #     "estimate_time_only": "empirical",
    #     "default_precision": 0.2,  # Default precision is applied to all pubs that don't have a precision specified, see API reference for more details
    #     "max_execution_time": 3600,  # You can specify a maximum QPU time in seconds, see API reference for more details
    #     "transpilation_level": "standard",  # "minimal_with_layout_opt" for minimal transpilation, see API reference for more details
    #     "parallel_execution": True,  # True for parallel execution, see API reference for more details
    # },
)

## 지원 받기

Qedma 지원팀이 도움을 줍니다! QESEM Qiskit Function을 사용하면서 문제가 발생하거나 질문이 있으시면 주저하지 마시고 연락하세요. 저희의 지식이 풍부하고 친절한 지원 직원이 기술적 우려나 문의 사항을 도와드릴 준비가 되어 있습니다.

도움을 받으시려면 support@qedma.com으로 이메일을 보내세요. 신속하고 정확한 답변을 제공할 수 있도록 경험하고 있는 문제에 대한 가능한 한 많은 세부 사항을 포함해 주세요. 전용 Qedma POC 담당자에게 이메일이나 전화로 연락할 수도 있습니다.

더 효율적으로 도움을 드리기 위해 연락하실 때 다음 정보를 제공해 주세요:

- 문제에 대한 상세한 설명
- Job ID
- 관련 오류 메시지 또는 코드

저희 Qiskit Function을 통해 최고의 경험을 보장하기 위해 신속하고 효과적인 지원을 제공하기 위해 노력하고 있습니다.

저희는 항상 제품을 개선하고 여러분의 제안을 환영합니다! 서비스를 향상시키거나 원하는 기능에 대한 아이디어가 있으시면 support@qedma.com으로 의견을 보내주세요.

## 다음 단계

> **Tip:** - [Qedma QESEM 접근 요청](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - 이 Qiskit Function의 [API 참조](https://docs.quantum.ibm.com/api/functions/qedma-qesem)를 방문하세요.
> - [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199)를 검토하세요.
> - [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997)를 검토하세요.

In [25]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)
print(sample_job.status())
sample_result = sample_job.result()

3ac6b2df-15b0-4dc0-8f48-cf14bd20a1c8
DONE


The following code snippet demonstrates how to retrieve the mitigation results and execution metrics. These contain essential data that enables a deeper understanding of how different parameters impact the QESEM execution. It may also be relevant when writing a paper based on your research.

In [50]:
for pub_idx, pub_result in enumerate(
    sample_result
):  # each element in the list is a result for a different pub, here we sent only one pub
    print(f"\nPUB {pub_idx}:")
    print(
        f"  The QPU time that this job took is (here it is 0 because we used fake_fez): {pub_result.metadata['total_qpu_time']}"
    )
    print(
        f"  Gates fidelity measured during the experiment: {pub_result.metadata['gate_fidelities']}"
    )
    print(f"  Total shots: {pub_result.metadata['total_shots']}")
    print(
        f"  Number of shots used for mitigation: {pub_result.metadata['mitigation_shots']}"
    )
    print(
        f"  Resource usage breakdown: {pub_result.metadata['resource_usage']}"
    )


PUB 0:
  The QPU time that this job took is (here it is 0 because we used fake_fez): 0.0
  Gates fidelity measured during the experiment: {'CZ': 0.9953704216147041, 'ID1Q': 0.9991834123567518}
  Total shots: 446000
  Number of shots used for mitigation: 194000
  Resource usage breakdown: {'RUNNING: MAPPING': {'CPU_TIME': 32.52745003718883, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 257.850521848537, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}


In `metadata["results"]`, results are grouped first by circuit instance and then by observable.

In [99]:
print("Full QESEM mitigation results:")

for pub_idx, pub_result in enumerate(sample_result):
    print(f"\nPUB {pub_idx}:")

    print(f"  Mitigated expectation values: {pub_result.data.evs}")
    print(f"  Mitigated error bars: {pub_result.data.stds}")
    noisy_results = pub_result.metadata.get("noisy_results")
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")

    print("  Transpiled circuits:")
    for circ_idx, transpiled_circ in enumerate(
        pub_result.metadata["transpiled_circs"]
    ):
        print(f"    Circuit {circ_idx}:")
        # print(f"      Circuit: \n {transpiled_circ['circuit']}") # not printing it because it's long but you can see the transpiled circuit itself
        print(f"      Qubit mapping: {transpiled_circ['qubit_maps']}")
        print(
            f"      Measurement bases: {transpiled_circ['num_measurement_bases']}"
        )

    print("  Results for each observable:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"    Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            print(
                f"        QESEM mitigated value: {result_dict['qesem']['value']} ± {result_dict['qesem']['error_bar']}"
            )
            print(
                f"        Unmitigated value: {result_dict['unmitigated']['value']} ± {result_dict['unmitigated']['error_bar']}"
            )

Full QESEM mitigation results:

PUB 0:
  Mitigated expectation values: [1.00648343 1.00636289]
  Mitigated error bars: [0.00253812 0.00693586]
  Unmitigated expectation values: [0.98031429 0.96357143]
  Unmitigated error bars: [0.00124128 0.00578812]
  Transpiled circuits:
    Circuit 0:
      Qubit mapping: [[[0, 140], [1, 141], [2, 142], [3, 143], [4, 136]]]
      Measurement bases: 2
  Results for each observable:
    Circuit 0:
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        QESEM mitigated value: 1.0064834305181962 ± 0.002538119914534849
        Unmitigated value: 0.9803142857142859 ± 0.0012412835813609938
      Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        QESEM mitigated value: 1.0063628870614818 ± 0.006935859820870656
        Unmitigated value: 0.9635714285714285 ± 0.005788121870526659


**Main results breakdown:**
- `mitigated`: the fully mitigated QESEM expectation value.
- `unmitigated`: the raw physical-noise result without error mitigation.

#### QESEM heuristic extrapolation results

In a standard QESEM run with a single `precision` float, the results also include automatically available noise-scaling points that are used for the QESEM heuristic. These points are computed without additional QPU resources.

Scale `1.0` represents the physical device noise level with readout mitigation (REM), while scale `2.0` is the complementary noise-amplified point, also with REM. These points are used to produce the `qesem_heuristic` result.

- `qesem_heuristic`: a ZNE-style estimate computed from the available noise-scaled data. Currently, this uses exponential extrapolation.
- `noise_scaling.results_with_REM`: expectation values at different noise scales, all with readout mitigation (REM).

A subtle but important detail is that the scale `1.0` result is not the same as the `unmitigated` result. Both correspond to the physical device noise level, but the scale `1.0` point includes readout mitigation, while `unmitigated` does not.

In [101]:
print("QESEM heuristic results:")

for pub_idx, pub_result in enumerate(sample_result):
    print(f"\nPUB {pub_idx}:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"  Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"    Observable {obs_idx}: {obs_array}")
            qesem_heuristic = result_dict["qesem_heuristic"][0]
            print("      QESEM heuristic:")
            print(
                f"        Value: {qesem_heuristic['value']} ± {qesem_heuristic['error_bar']}"
            )
            print(
                f"        Extrapolation: {qesem_heuristic['extrapolation']}"
            )
            print(
                f"        Scale factors: {qesem_heuristic['scale_factors']}"
            )
            noise_scaling = result_dict["noise_scaling"]
            print("      Noise scaling results:")
            print(
                f"        Scaling method: {noise_scaling['scaling_method']}"
            )
            print("        Results with Readout mitigation (REM):")
            for rem_result in sorted(
                (
                    item
                    for item in noise_scaling["results_with_REM"]
                    if item["scale"] != 0.0
                ),
                key=lambda item: item["scale"],
            ):
                print(
                    f"          Scale factor {rem_result['scale']}: {rem_result['value']} ± {rem_result['error_bar']}"
                )

QESEM heuristic results:

PUB 0:
  Circuit 0:
    Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
      QESEM heuristic:
        Value: 1.0008161638888535 ± 0.0038859458884403964
        Extrapolation: exponential
        Scale factors: [1.0, 2.0]
      Noise scaling results:
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 1.0: 0.9918395459270772 ± 0.0012565417579355634
          Scale factor 2.0: 0.982943441922748 ± 0.0028919278067695018
    Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
      QESEM heuristic:
        Value: 0.9960853148925298 ± 0.013811635038961175
        Extrapolation: exponential
        Scale factors: [1.0, 2.0]
      Noise scaling results:
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 1.0: 0.9902860583785115 ± 0.005921236914723409
          Scale factor 2.0: 0.9845205654

<Admonition type="note">
The following examples focus on feature-specific inputs and results, so they do not print the full execution metrics each time. The top-level metadata shown earlier, such as <code>total_qpu_time</code>, <code>gate_fidelities</code>, <code>total_shots</code>, <code>mitigation_shots</code>, and <code>resource_usage</code>, is available for these jobs as well.

Some variables from the previous examples, including the backend, observables, and base circuit parameters, are reused below for conciseness.
</Admonition>

<Admonition type="note">
All of the following examples can also be run with empirical time estimation. To enable it, pass <code>"estimate_time_only": "empirical"</code> in the function options.
</Admonition>

### Parameterized circuit example

Many algorithms evaluate the same circuit at several parameter values. Sending the parameterized circuit as one QESEM job lets QESEM share characterization and calibration across the circuit instances, which can reduce QPU-time overhead compared with running separate jobs.

Submitting a parameterized circuit requires using the `"minimal_with_layout_opt"` transpilation level.
Circuits submitted at this level should already be expressed using the backend's basis gates, depending on the backend. At this level, QESEM keeps the submitted structure as close as possible to the input circuit, respects barriers during layerification (grouping operations into layers of parallel two-qubit gates), and still handles hardware mapping to high-fidelity qubits and device connectivity automatically.

In practice, this means you should transpile circuits to the target backend’s basis gates before submission. A simple basis-gate transpilation example is shown below.

QESEM currently supports only one observable per parameter set. The two parameter rows below are zipped with the two observables: the first row is measured with `avg_magnetization`, and the second row is measured with `other_observable`.

In [ ]:
# Transpile to the backend basis gates only. With minimal_with_layout_opt, QESEM handles hardware mapping/connectivity and observable layout internally.
from qiskit_ibm_runtime.fake_provider import FakeFez

backend = FakeFez()
basis = backend.operation_names
print(basis)

param0 = qiskit.circuit.Parameter("param0")
param1 = qiskit.circuit.Parameter("param1")
parametrized_circ = qiskit.QuantumCircuit(5)
parametrized_circ.rx(param0, 0)
parametrized_circ.rx(param1, 1)
parametrized_circ.cx(0, 1)
parametrized_circ.cx(2, 3)
parametrized_circ.cx(1, 2)
parametrized_circ.cx(3, 4)

parametrized_circ = qiskit.transpile(
    parametrized_circ, basis_gates=basis, optimization_level=1
)
parametrized_parameter_values = [[0.5, 0.1], [0.0, 0.6]]
parametrized_observables = [avg_magnetization, other_observable]

parametrized_job = qesem_function.run(
    pubs=[
        (
            parametrized_circ,
            parametrized_observables,
            parametrized_parameter_values,
            0.1,
        )
    ],
    backend_name=backend_name,
    options={
        "max_execution_time": 300,
        "transpilation_level": "minimal_with_layout_opt",
    },
)

In [72]:
print(parametrized_job.job_id)
print(parametrized_job.status())

d1b0e29b-196c-4896-aec6-44a268ebd874
DONE


In [73]:
parametrized_result = parametrized_job.result()

In [77]:
print("Parameterized circuit QESEM results:")

for pub_idx, pub_result in enumerate(parametrized_result):
    print(f"\nPUB {pub_idx}:")
    print(f"  Mitigated expectation values: {pub_result.data.evs}")
    print(f"  Mitigated error bars: {pub_result.data.stds}")
    noisy_results = pub_result.metadata["noisy_results"]
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")
    print("  Results for each parameter value:")
    for param_idx, param_results in enumerate(pub_result.metadata["results"]):
        print(
            f"    Parameter set {param_idx}: {parametrized_parameter_values[param_idx]}"
        )
        for obs_idx, (obs_array, result_dict) in enumerate(param_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            print(
                f"        QESEM mitigated value: {result_dict['qesem']['value']} ± {result_dict['qesem']['error_bar']}"
            )
            print(
                f"        Unmitigated value: {result_dict['unmitigated']['value']} ± {result_dict['unmitigated']['error_bar']}"
            )

Parameterized circuit QESEM results:

PUB 0:
  Mitigated expectation values: [0.92392021 0.82517653]
  Mitigated error bars: [0.00565281 0.00616016]
  Unmitigated expectation values: [0.9028     0.78771429]
  Unmitigated error bars: [0.00335142 0.00925413]
  Results for each parameter value:
    Parameter set 0: [0.5, 0.1]
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        QESEM mitigated value: 0.923920210615709 ± 0.005652811570890183
        Unmitigated value: 0.9028 ± 0.0033514176105045447
    Parameter set 1: [0.0, 0.6]
      Observable 0: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        QESEM mitigated value: 0.8251765289285893 ± 0.006160161743353999
        Unmitigated value: 0.7877142857142858 ± 0.009254130564027902


### Multi-pub example

Multi-pub execution is useful when you want to run several related circuits in one QESEM job. Like parameterized execution, this can reduce overhead because QESEM can share characterization and calibration across circuit instances instead of repeating it in separate jobs.

This is especially useful for circuits with a shared layer structure, such as **Trotter-type** workloads, where different circuits reuse the same unique layers. In that case, running them together can reduce characterization cost compared with independent QESEM jobs.

Multi-pub jobs require `"transpilation_level": "minimal_with_layout_opt"`. As in the parameterized example, the circuits should be transpiled to the target backend’s basis gates before submission. QESEM then handles device connectivity, layout, and mapping to high-fidelity qubits internally.

Each PUB below contains one circuit and the same two observables used earlier in the notebook, so the returned `PrimitiveResult` contains one `PubResult` per input circuit.

The example below uses two simple Trotter circuits with the same layer pattern: `circ_a` has one Trotter layer, and `circ_b` repeats the same layer pattern twice. This makes the shared structure explicit.

In [ ]:
def make_trotter_circuit(num_qubits, num_layers, zz_angle=0.2, x_angle=0.1):
    trotter_circ = qiskit.QuantumCircuit(num_qubits)
    for _ in range(num_layers):
        for q in range(num_qubits):
            trotter_circ.rx(x_angle, q)
        trotter_circ.barrier()
        for q in range(0, num_qubits - 1, 2):
            trotter_circ.rzz(zz_angle, q, q + 1)
        trotter_circ.barrier()
        for q in range(1, num_qubits - 1, 2):
            trotter_circ.rzz(zz_angle, q, q + 1)
        trotter_circ.barrier()
    return trotter_circ


circ_a = make_trotter_circuit(num_qubits=5, num_layers=1)
circ_b = make_trotter_circuit(num_qubits=5, num_layers=2)

In [ ]:
multi_pubs = [
    (
        qiskit.transpile(qci, basis_gates=basis, optimization_level=1),
        [avg_magnetization, other_observable],
    )
    for qci in [circ_a, circ_b]
]

multi_circ_job = qesem_function.run(
    pubs=multi_pubs,
    backend_name=backend_name,
    options={
        "max_execution_time": 300,
        "transpilation_level": "minimal_with_layout_opt",
        "default_precision": 0.1,
    },
)

In [91]:
print(multi_circ_job.job_id)
print(multi_circ_job.status())

e34565b8-7262-4133-a120-de42ce624a99
DONE


In [92]:
multi_circ_result = multi_circ_job.result()

In [105]:
print("Multi-pub QESEM results:")

for pub_idx, pub_result in enumerate(multi_circ_result):
    print(f"\nPUB {pub_idx}:")
    print(f"  Mitigated expectation values: {pub_result.data.evs}")
    print(f"  Mitigated error bars: {pub_result.data.stds}")
    noisy_results = pub_result.metadata["noisy_results"]
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")
    print("  Results for each observable:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"    Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            print(
                f"        QESEM mitigated value: {result_dict['qesem']['value']} ± {result_dict['qesem']['error_bar']}"
            )
            print(
                f"        Unmitigated value: {result_dict['unmitigated']['value']} ± {result_dict['unmitigated']['error_bar']}"
            )

Multi-pub QESEM results:

PUB 0:
  Mitigated expectation values: [0.99502406 1.02209332]
  Mitigated error bars: [0.00483819 0.00707488]
  Unmitigated expectation values: [0.96934286 0.97271429]
  Unmitigated error bars: [0.00124855 0.00617294]
  Results for each observable:
    Circuit 0:
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        QESEM mitigated value: 0.9950240647925642 ± 0.004838188086259301
        Unmitigated value: 0.9693428571428573 ± 0.0012485470362492692
      Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        QESEM mitigated value: 1.0220933230604674 ± 0.007074884384355636
        Unmitigated value: 0.9727142857142859 ± 0.006172939375512439

PUB 1:
  Mitigated expectation values: [0.98850017 1.02555188]
  Mitigated error bars: [0.0077912  0.01672652]
  Unmitigated expectation values: [0.93682857 0.95371429]
  Unmitigated error bars: [0.00156245 0.00665735]
  Result

### Quasi-probabilistic Error Tuning (QET) example

Quasi-probabilistic Error Tuning (QET) requests expectation values at selected noise scale factors. This is useful for custom noise-scaling studies and Zero-Noise Extrapolation workflows. Scale `1.0` is the physical noise level, values between `0.0` and `1.0` partially reduce noise, and values above `1.0` amplify noise.

To use QET with the Qiskit Function, pass a dictionary as the PUB precision. The dictionary maps each requested noise scale to its target precision. Returned scale-factor results are stored in `noise_scaling.results_with_REM` and include readout mitigation. The scale `1.0` point is therefore not identical to the unmitigated value, because `1.0` includes readout mitigation while `unmitigated` does not.

When a scale is requested, QESEM also returns the complementary scale around `1.0` without extra QPU usage. For example, requesting `0.5` can also return `1.5`, and requesting `1.3` can also return `0.7`. The precision on the complementary scale is not guaranteed.

In [88]:
noise_scale_precision = {0.5: 0.15, 1.3: 0.2}

qet_job = qesem_function.run(
    pubs=[
        (
            circ,
            [avg_magnetization, other_observable],
            None,
            noise_scale_precision,
        )
    ],
    backend_name=backend_name,
    options={"max_execution_time": 300},
)

In [95]:
print(qet_job.job_id)
print(qet_job.status())

8195fa58-f037-4651-8715-36ce1cdc5521
DONE


In [96]:
qet_result = qet_job.result()

In [104]:
print("QET noise-scaling results:")

for pub_idx, pub_result in enumerate(qet_result):
    print(f"\nPUB {pub_idx}:")
    noisy_results = pub_result.metadata["noisy_results"]
    print(f"  Unmitigated expectation values: {noisy_results.evs}")
    print(f"  Unmitigated error bars: {noisy_results.stds}")
    print("  Results for each observable:")
    for circ_idx, circ_results in enumerate(pub_result.metadata["results"]):
        print(f"    Circuit {circ_idx}:")
        for obs_idx, (obs_array, result_dict) in enumerate(circ_results):
            print(f"      Observable {obs_idx}: {obs_array}")
            noise_scaling = result_dict["noise_scaling"]
            print(
                f"        Scaling method: {noise_scaling['scaling_method']}"
            )
            print("        Results with Readout mitigation (REM):")
            for rem_result in sorted(
                (
                    item
                    for item in noise_scaling["results_with_REM"]
                    if item["scale"] != 0.0
                ),
                key=lambda item: item["scale"],
            ):
                print(
                    f"          Scale factor {rem_result['scale']}: {rem_result['value']} ± {rem_result['error_bar']}"
                )

QET noise-scaling results:

PUB 0:
  Unmitigated expectation values: [0.97822857 0.96171429]
  Unmitigated error bars: [0.00123812 0.00672958]
  Results for each observable:
    Circuit 0:
      Observable 0: ObservablesArray({'IIIIZ': 0.2, 'IIIZI': 0.2, 'IIZII': 0.2, 'IZIII': 0.2, 'ZIIII': 0.2}, shape=())
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 0.5: 0.9938730340199383 ± 0.0032116907357568275
          Scale factor 0.7: 0.9963976191853445 ± 0.00036300258869586616
          Scale factor 1.0: 0.9898115079506586 ± 0.0012525947426560995
          Scale factor 1.3: 0.9864667065580341 ± 0.002633221613518526
          Scale factor 1.5: 0.9838755527197551 ± 0.002948417797996015
      Observable 1: ObservablesArray({'IIIZZ': 1.0, 'ZIIXI': 0.5}, shape=())
        Scaling method: QESEM
        Results with Readout mitigation (REM):
          Scale factor 0.5: 1.0006538544450332 ± 0.002121742014777343
          Scale factor 0.7: 1.0004159

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com.

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
- Visit the [API reference](/docs/api/functions/qedma-qesem) for this Qiskit Function.
- Try the [Simulate 2D tilted-field Ising with the QESEM function](/docs/tutorials/qedma-2d-ising-with-qesem) tutorial.
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).
- Review [Aharonov, D., et al. (2025). Syndrome aware mitigation of logical errors. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).
- Review [Aharonov, D., et al. (2025). On the Importance of Error Mitigation for Quantum Computation. arXiv preprint arXiv:2512.23810](https://arxiv.org/abs/2512.23810).
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
- Review [Goldack, M., et al. (2026). Computing Statistical Properties of Velocity Fields on Current Quantum Hardware. arXiv preprint arXiv:2601.10166](https://arxiv.org/pdf/2601.10166).
- Review [Sakuma, R., et al. (2026). Point-group symmetry analysis of many-electron wavefunctions on a quantum computer arXiv preprint arXiv:2605.24824](https://arxiv.org/abs/2605.24824).

</Admonition>